## Introduction
#### Data: Meta AI Fair Speech Dataset -- An open-source audio dataset of 26,471 utterances from 593 people in the US who were paid to record and submit audio of themselves saying commands in response to prompts that relate to music, capture, utilities, notification control, messaging, calling, and dictation. For example, 'How would you search for a song?' This dataset was designed with speech recognition for digital assistants in mind as its intended application. I randomly sampled 100, <60sec utterances for each of my three groups of interest.
#### API -- Google's Chirp 3 API -- An open-source multilinguial speech model that is marketed for transcription and speech synthesis applications.
#### Hypothesis -- The speech-to-text model will have a higher word error rate for English as a Second Language speakers whose first language is underrepresented in the dataset compared to English, particularly Hindi and Urdu.
#### Motivation -- Speech-to-text systems should work for a variety of users in order to be effective and fair applications, including individuals whose first language was not English and may speak with an accent, but it's unclear what the makeup of the training datasets are that power these high profile models. I'd like to test how accurate a common open-source speech-to-text model is for individuals whose first language is highly represented in the global population but not US demographics, such as Hindi (865k speakers in the US vs. 600m worldwide) and Urdu (508k speakers in the US vs. 230m worldwide). Additionally, I'm interested in how a model trained by one large tech company will perform on a dataset from a different company, in this case Google and Meta, respectively.

#### Call API to transcribe speech

In [ ]:
from google.cloud import speech

def transcribe(filepaths:list):
    # Instantiates a client
    client = speech.SpeechClient()

    # The name of the audio filepath
    gcs_uri = "gs://asr_fairness_audio_data/"

    # model config
    config = speech.RecognitionConfig(
        encoding=speech.RecognitionConfig.AudioEncoding.LINEAR16,
        sample_rate_hertz=16000,
        language_code="en-US",
    )

    # store output
    outs = []

    print("Transcribing...")
    for f in filepaths:
        # transcription
        audio = speech.RecognitionAudio(uri=gcs_uri+f)
        response = client.recognize(config=config, audio=audio)

        # concatenate transcription and add to output
        curr_out = ""
        for result in response.results:
            curr_out += result.alternatives[0].transcript
        outs.append(curr_out)

    print("Done!")
    return outs

In [14]:
import pandas as pd

# get input files and metadata
metadata = pd.read_csv("audio_subset/subset_metadata.csv")
trans_true = metadata["transcription"].tolist()

# filenames for passing to func
filenames = [f"{f}.wav" for f in metadata["hash_name"].tolist()]

# generate transcriptions
trans_preds = transcribe(filenames)

Transcribing...
Done!


In [15]:
len(trans_preds)

300

In [16]:
trans_preds

['hey Facebook shared a video to my father',
 'hi there I am very much frustrated about these screens and like uh I am having like bugs problem in my house but it is not getting done like soon and my kids also having trouble with this so please get it done as soon as possible thank you and the other way is I am really frustrated how slow your maintenance procedure is I am sending you email messages and all and I am not getting any solution for it thank you',
 "hi love I was just um talking to my friend and uh we were just discussing random things and she just mentioned the fact that it is impossible for most people to touch their elbows with their tongue um and I found it so funny let's try it with the kids tonight and see if any of us can touch our elbows with our tongue by",
 'reply back to George II saying hey random fact there is no way that baby whales consume 200 liters of milk a day finish text message send',
 'hi uh you just called me and I was really shocked that you lost your

In [17]:
# save outputs
model_output = metadata.copy()
model_output["model_transcription"] = trans_preds
model_output.to_csv("audio_subset/full_data.csv",index=False)

#### Analyze results

#### Discussion
##### 

#### Citations
- Meta AI Fair Speech Dataset. https://ai.meta.com/datasets/speech-fairness-dataset/
- Google Cloud Speech to Text. https://docs.cloud.google.com/speech-to-text/docs/overview#:~:text=for%20feature%20availability.-,Model%20selection,select%20one%20in%20your%20requests.
- Languages of the US. https://en.wikipedia.org/wiki/Languages_of_the_United_States
- 10 most spoken languages in the world in 2025. https://www.icls.edu/blog/most-spoken-languages-in-the-world
- Improving fairness and robustness in speech recognition. https://ai.meta.com/blog/improving-fairness-and-robustness-in-speech-recognition/
- code for API call modified from tutorial for Google Cloud's Speech-to-Text